# FinNexus — XGBoost Test / Evaluation Notebook

Loads trained `.pkl` models and evaluates them on held-out test data.

**Data:** `Test/<AssetClass>/<SYMBOL>_features.csv`  
**Models:** `Data/XGBoost_Results/models/<AssetClass>/<SYMBOL>_xgb.pkl`  
**Outputs:**
- Confusion matrices per asset
- ROC curves per asset class
- `Data/XGBoost_Results/test_summary.csv`
- `Data/XGBoost_Results/xgb_test_report.txt`


## 1. Setup

In [ ]:
import pickle, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, RocCurveDisplay, classification_report
)
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

ROOT       = Path('..').resolve()
TEST_DIR   = ROOT / 'Test'
MODEL_DIR  = ROOT / 'Data' / 'XGBoost_Results' / 'models'
OUTPUT_DIR = ROOT / 'Data' / 'XGBoost_Results'
CM_DIR     = OUTPUT_DIR / 'confusion_matrices'
ROC_DIR    = OUTPUT_DIR / 'roc_curves'

for d in [CM_DIR, ROC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_COL   = 'target_7d'
ASSET_CLASSES = ['Crypto', 'ETFs', 'Commodities', 'Futures', 'Stocks']

DROP_COLS = [
    'Date', 'Open', 'High', 'Low', 'Close', 'Volume',
    'is_outlier', 'data_quality',
    'target_3d', 'target_5d', 'target_7d', 'target_10d', 'target_14d'
]

print(f'ROOT      : {ROOT}')
print(f'TEST_DIR  : {TEST_DIR}')
print(f'MODEL_DIR : {MODEL_DIR}')

## 2. Discover Test Files & Models

In [ ]:
def find_test_model_pairs(test_dir, model_dir, asset_classes):
    """Returns [(test_csv, model_pkl, asset_class, symbol), ...]."""
    pairs = []
    for ac in asset_classes:
        test_ac_dir = test_dir / ac
        model_ac_dir = model_dir / ac
        if not test_ac_dir.exists() or not model_ac_dir.exists():
            continue
        for test_csv in test_ac_dir.glob('*_features.csv'):
            symbol = test_csv.stem.replace('_features', '')
            model_pkl = model_ac_dir / f'{symbol}_xgb.pkl'
            if model_pkl.exists():
                pairs.append((test_csv, model_pkl, ac, symbol))
    return pairs

pairs = find_test_model_pairs(TEST_DIR, MODEL_DIR, ASSET_CLASSES)
print(f'Found {len(pairs)} test/model pairs')

if not pairs:
    raise RuntimeError('No test data or models found. Run training notebook first.')

# Preview
print('\nSample pairs:')
for i in range(min(5, len(pairs))):
    _, _, ac, sym = pairs[i]
    print(f'  {ac:15s} / {sym}')

## 3. Preprocessing Helper

In [ ]:
def load_test_data(csv_path, target_col, drop_cols, feature_names):
    """
    Load test CSV, align columns to match training feature_names,
    impute missing, return (X, y).
    """
    df = pd.read_csv(csv_path, parse_dates=['Date'])
    df.sort_values('Date', inplace=True)
    df.reset_index(drop=True, inplace=True)

    if target_col not in df.columns:
        raise ValueError(f'{target_col} missing in {csv_path.name}')

    y = df[target_col].values.astype(int)

    # Drop non-features
    to_drop = [c for c in drop_cols if c in df.columns]
    X_df = df.drop(columns=to_drop)
    X_df = X_df.select_dtypes(include=[np.number])

    # Align to training features (add missing columns as 0)
    for feat in feature_names:
        if feat not in X_df.columns:
            X_df[feat] = 0.0
    X_df = X_df[feature_names]

    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X_df.values).astype(np.float32)
    return X, y

print('Preprocessing helper defined.')

## 4. Evaluation Loop

In [ ]:
test_results = []

for test_csv, model_pkl, asset_class, symbol in pairs:
    try:
        with open(model_pkl, 'rb') as f:
            obj = pickle.load(f)
        model = obj['model']
        meta  = obj['meta']
        feat_names = meta['feature_names']

        X_test, y_test = load_test_data(test_csv, TARGET_COL, DROP_COLS, feat_names)

        if len(y_test) < 10 or len(np.unique(y_test)) < 2:
            print(f'SKIP {symbol} — not enough test data')
            continue

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        test_results.append({
            'asset_class': asset_class,
            'symbol'     : symbol,
            'n_test'     : len(y_test),
            'accuracy'   : accuracy_score(y_test, y_pred),
            'f1'         : f1_score(y_test, y_pred, zero_division=0),
            'auc'        : roc_auc_score(y_test, y_prob),
            'precision'  : float(np.mean(y_pred == y_test)),  # simplified
        })

    except Exception as e:
        print(f'ERROR {asset_class}/{symbol}: {e}')

test_df = pd.DataFrame(test_results).sort_values('auc', ascending=False)
print(f'\nEvaluated {len(test_df)} models on test sets.')
display(test_df.head(10))

## 5. Confusion Matrices

In [ ]:
def plot_confusion_matrix(model, X_test, y_test, title, save_path):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Down','Up'], yticklabels=['Down','Up'], ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()

# Plot top 3 per asset class
for asset_class in ASSET_CLASSES:
    sub = test_df[test_df.asset_class == asset_class].head(3)
    for _, row in sub.iterrows():
        pkl = MODEL_DIR / asset_class / f"{row['symbol']}_xgb.pkl"
        csv = TEST_DIR / asset_class / f"{row['symbol']}_features.csv"
        if not pkl.exists() or not csv.exists():
            continue
        with open(pkl, 'rb') as f:
            obj = pickle.load(f)
        X_t, y_t = load_test_data(csv, TARGET_COL, DROP_COLS, obj['meta']['feature_names'])
        save_path = CM_DIR / f"{asset_class}_{row['symbol']}_cm.png"
        plot_confusion_matrix(
            obj['model'], X_t, y_t,
            title=f"{asset_class} / {row['symbol']}  AUC={row['auc']:.3f}",
            save_path=save_path,
        )
print('Confusion matrices saved.')

## 6. ROC Curves per Asset Class

In [ ]:
for asset_class in ASSET_CLASSES:
    sub = test_df[test_df.asset_class == asset_class]
    if sub.empty:
        continue

    fig, ax = plt.subplots(figsize=(7, 5))
    for _, row in sub.iterrows():
        pkl = MODEL_DIR / asset_class / f"{row['symbol']}_xgb.pkl"
        csv = TEST_DIR / asset_class / f"{row['symbol']}_features.csv"
        if not pkl.exists() or not csv.exists():
            continue
        with open(pkl, 'rb') as f:
            obj = pickle.load(f)
        X_t, y_t = load_test_data(csv, TARGET_COL, DROP_COLS, obj['meta']['feature_names'])
        if len(np.unique(y_t)) < 2:
            continue
        RocCurveDisplay.from_estimator(
            obj['model'], X_t, y_t,
            name=f"{row['symbol']} (AUC={row['auc']:.3f})",
            ax=ax, alpha=0.75,
        )

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    ax.set_title(f'ROC Curves — {asset_class}', fontsize=12)
    ax.legend(fontsize=7, loc='lower right')
    plt.tight_layout()
    roc_path = ROC_DIR / f'{asset_class}_roc.png'
    plt.savefig(roc_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {roc_path.name}')

## 7. Comparative Metrics Dashboard

In [ ]:
# Per-class averages
agg_test = test_df.groupby('asset_class')[['accuracy','f1','auc']].mean().round(4)
print('── Test Metrics by Asset Class ──')
display(agg_test)

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, colour in zip(axes, ['accuracy','f1','auc'], ['steelblue','coral','forestgreen']):
    agg_test[metric].plot.bar(ax=ax, color=colour, edgecolor='white', rot=30)
    ax.set_title(f'Test {metric.upper()}', fontsize=11)
    ax.set_ylim(0, 1)
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=8
        )
plt.suptitle('FinNexus XGBoost — Test Performance by Asset Class', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_metrics_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Test Report

In [ ]:
report = [
    '=' * 60,
    'FinNexus XGBoost Test Report',
    f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'Target    : {TARGET_COL}',
    '=' * 60,
    f'Total models evaluated : {len(test_df)}',
    '',
]

for ac in ASSET_CLASSES:
    sub = test_df[test_df.asset_class == ac]
    if sub.empty:
        continue
    report.append(f'[{ac}]  {len(sub)} assets')
    report.append(f'  Avg Accuracy : {sub.accuracy.mean():.4f}')
    report.append(f'  Avg F1       : {sub.f1.mean():.4f}')
    report.append(f'  Avg AUC      : {sub.auc.mean():.4f}')
    best = sub.sort_values('auc', ascending=False).iloc[0]
    report.append(f'  Best model   : {best.symbol}  AUC={best.auc:.4f}')
    report.append('')

report += [
    '── Top 10 by Test AUC ──',
    test_df[['asset_class','symbol','auc','f1','accuracy','n_test']].head(10).to_string(index=False),
    '',
    '=' * 60,
]

report_text = '\n'.join(report)
(OUTPUT_DIR / 'xgb_test_report.txt').write_text(report_text)
test_df.to_csv(OUTPUT_DIR / 'test_summary.csv', index=False)

print(report_text)
print(f'\nTest summary saved: {OUTPUT_DIR / "test_summary.csv"}')